In [1]:
%%capture
%pip install -U transformers 
%pip install -U datasets 
%pip install -U accelerate 
%pip install -U peft 
%pip install -U trl 
%pip install -U bitsandbytes 
%pip install -U wandb

In [2]:
import huggingface_hub
print(huggingface_hub.__version__)


0.28.1


In [3]:
!pip install --no-cache-dir --upgrade huggingface_hub


In [4]:
from huggingface_hub.utils import OfflineModeIsEnabled


In [5]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/conda/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning

In [6]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HugFace")

login(token = hf_token)

wb_token = user_secrets.get_secret("wandb")

wandb.login(key=wb_token)
run = wandb.init(
    project='Fine-tune Mistral 7B Instruct on Medical Dataset', 
    job_type="training", 
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: trisha-sengupta-ece26 (trisha-sengupta-ece26-Heritage Institute of Technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Tracking run with wandb version 0.19.5
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250202_122352-vfush6pv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run devoted-sunset-3
wandb: ⭐️ View project at https://wandb.ai/trisha-sengupta-ece26-Heritage%20Institute%20of%20Technology/Fine-tune%20Mistral%207B%20Instruct%20on%20Medical%20Dataset?apiKey=a35cf0b0dd001e9ea8a077044492b7

In [7]:
base_model = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1"
dataset1 = load_dataset("Amod/mental_health_counseling_conversations")

new_model = "llama-3-8b-chat-doctor"

README.md:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

combined_dataset.json:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [8]:
torch_dtype = torch.float16
attn_implementation = "eager"

In [9]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [10]:
from datasets import  Dataset
df = dataset1['train'].to_pandas()

# Reduce the number of rows
df_reduced = df.sample(n=1000, random_state=42)

# Convert back to Hugging Face dataset
dataset_reduced =  Dataset.from_pandas(df_reduced)

In [11]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.padding_side = 'right'
tokenizer.chat_template = None
model, tokenizer = setup_chat_format(model, tokenizer)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [12]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)
model = get_peft_model(model, peft_config)

In [13]:
dataset_reduced

Dataset({
    features: ['Context', 'Response', '__index_level_0__'],
    num_rows: 1000
})

In [14]:
def format_chat_template(row):
    row_json = [{"role": "user", "content": row["Context"]},
               {"role": "assistant", "content": row["Response"]}]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

In [15]:



dataset = dataset_reduced.map(
    format_chat_template,
    num_proc=4,
)

dataset['text'][3]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

"<|im_start|>user\nCheating is something unacceptable for me but because we have two daughters I decided not to break up the family. However, now I am struggling to forget and forgive what happened. I feel like I cannot trust him. Without trust, I cannot stay in this relationship. On the other hand, I do not want my children to get hurt. I'm not sure how to move forward?<|im_end|>\n<|im_start|>assistant\nIt is completely understandable that you are struggling to forgive and forget this betrayal, and I'd like to echo the sentiment of Danielle Alvarez: infidelity takes time to heal from, so allow yourself to grieve and find the support you need. I'd highly suggest going to couples therapy and addressing all the issues that Danielle raised, especially whether he has expressed genuine remorse and is being completely transparent with you and is taking responsibility for the choice he made, including acknowledging the immense impact it had on you, your relationship, and your ability to trust

In [16]:
dataset = dataset_reduced.train_test_split(test_size=0.1)

In [17]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    evaluation_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="wandb"
)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [18]:
dataset = dataset.remove_columns(["Context", "Response", "__index_level_0__"])


In [19]:
print(dataset_reduced)

Dataset({
    features: ['Context', 'Response', '__index_level_0__'],
    num_rows: 1000
})


In [20]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
# Preprocess the dataset
def preprocess_function(examples):
    inputs = examples["Context"]
    targets = examples["Response"]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=512)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_reduced.map(preprocess_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [21]:
from sklearn.model_selection import train_test_split

# Manually split the dataset into train and test
train_dataset, test_dataset = train_test_split(tokenized_datasets, test_size=0.2)

# Now, you can use these datasets for training and evaluation


In [22]:
dataset["test"]

Dataset({
    features: [],
    num_rows: 100
})

In [23]:
from datasets import Dataset

# Convert the split dictionaries back to Dataset objects
train_dataset = Dataset.from_dict(train_dataset)
test_dataset = Dataset.from_dict(test_dataset)

In [24]:
trainer = SFTTrainer(
    model=model,
     train_dataset=train_dataset,  # Use the train split you created
    eval_dataset=test_dataset,  
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_arguments,
   
)

/tmp/ipykernel_26/2332616770.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(


In [25]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
80,1.547300,1.890117
160,1.773400,1.658569
240,2.222200,1.471239
320,1.604600,1.337128
400,1.812800,1.273746


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embed

TrainOutput(global_step=400, training_loss=1.669238642025739, metrics={'train_runtime': 3445.5158, 'train_samples_per_second': 0.232, 'train_steps_per_second': 0.116, 'total_flos': 1.75782575996928e+16, 'train_loss': 1.669238642025739, 'epoch': 1.0})

In [26]:
wandb.finish()
model.config.use_cache = True

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:               eval/loss █▅▃▂▁
wandb:            eval/runtime ▅▃█▁▁
wandb: eval/samples_per_second ▁▁▁▁▁
wandb:   eval/steps_per_second ▁▁▁▁▁
wandb:             train/epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇████
wandb:       train/global_step ▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▆▆▆▆▆▆▆▆▇▇▇▇███
wandb:         train/grad_norm ▆▇▅▆▅▆▅▆▂▅▂▆▅▁▇▄▅▆▄▄▆▅▆▃▇▃▅▇▃▂▆▅▆▃▆▂▁█▄▇
wandb:     train/learning_rate ▅▇█▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁
wandb:              train/loss ▆▇▇▇▆██▄▇▅▅▇▅▆▆▄▆▁▇▅▇▂▇▆▄▇▆▆▇▃▇▅▇▇▇▇▆▆▃▂
wandb: 
wandb: Run summary:
wandb:                eval/loss 1.27375
wandb:             eval/runtime 262.6111
wandb:  eval/samples_per_second 0.762
wandb:    eval/steps_per_second 0.762
wandb:               total_flos 1.75782575996928e+16
wandb:              train/epoch 1
wandb:        train/global_step 400
wandb:          train/grad_norm 4.11173
wandb:      train/learning_ra

### **Mental Health**

In [27]:
messages = [
    {
        "role": "user",
        "content": "I often feel anxious in social situations. What are some ways to manage anxiety without medication?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)
print(outputs)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

tensor([[    1,     1,   733, 16289, 28793,   315,  2608,  1601, 19695,   297,
          2809, 11846, 28723,  1824,   460,   741,  4342,   298,  8594, 12758,
          1671, 20859, 28804,   733, 28748, 16289, 28793,  1387,   460,  2856,
          4342,   298,  8594, 12758,  1671, 20859, 28747,    13,    13, 28740,
         28723, 14972, 14232, 20858, 28747,  8890,  3944, 28725,  3534,  5276,
         28713,   297,  1059,   574,  9948,   304,   575,  1059,   574,  5108,
         28723,   851,   541,  1316,  7643,  9388,   302, 12758,   304,  6727,
         28723,    13,    13, 28750, 28723, 14683, 19965, 23976, 28747, 25508,
          1250,  2169,   297,   272,  2470,   486, 18319,   356,   574,  7403,
         28725,  9388, 28725,   304,  7646,  1106,  3564,   697,  1671, 16548,
         28723,    13,    13, 28770, 28723, 19310,   495, 14540, 27607, 28747,
           320,  1058,   304,   868,  8096,  1430, 14540,  2071,   297,   574,
          2187,   624,   438,   264,   727, 28723,  

In [28]:
messages = [
    {
        "role": "user",
        "content": "I feel depressed and useless "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=False)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250,num_return_sequences=1,   num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

[INST] I feel depressed and useless  [/INST] I'm sorry to hear that you're feeling this way. It's important to remember that everyone experiences feelings of sadness and hopelessness from time to time. However, if these feelings are persisting and interfering with your daily life, it may be helpful to speak with a mental health professional. They can provide you with tools and techniques to help manage your symptoms and improve your overall well-being. In the meantime, there are a few things you can try to help yourself feel better:
  1. Get enough sleep: Lack of sleep can exacerbate feelings of depression. Try to establish a regular sleep schedule and create a relaxing bedtime routine.
  2. Exercise regularly: Physical activity has been shown to improve mood and reduce stress. Find an activity that you enjoy and try to do it at least three times per week.
  3. Eat a healthy diet: Eating a balanced diet can help improve your overall health and mood. Try to eat plenty of fruits, vegetab

In [29]:
messages = [
    {
        "role": "user",
        "content": "prescibe some good music and yogas for mental well being "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

[INST] prescibe some good music and yogas for mental well being  [/INST] There are many different types of music that can be beneficial for mental well-being. Here are a few examples:

1. Classical music: Studies have shown that listening to classical music, such as Mozart or Beethoven, can improve focus and productivity.
2. Nature sounds: Listening to the sounds of nature, such as birds chirping or waves crashing, can help reduce stress and promote relaxation.
3. Ambient music: Ambient music is a type of instrumental music that is designed to be calming and soothing. It can be helpful for activities such as meditation or yoga.
4. Pop music: Some people find that listening to upbeat pop music can help boost their mood and energy levels.

As for yoga, there are many different types of yoga that can be beneficial for mental well-being. Here are a few examples:

1. Hatha yoga: Hatha yoga is a gentle form of yoga that focuses on physical postures and breathing exercises. It can be helpful 

In [30]:

trainer.push_to_hub()

/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/692M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/trisha2710/llama-3-8b-chat-doctor/commit/5015f53dc18aa3b3616ba8c29bb6a89c900d92f7', commit_message='End of training', commit_description='', oid='5015f53dc18aa3b3616ba8c29bb6a89c900d92f7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/trisha2710/llama-3-8b-chat-doctor', endpoint='https://huggingface.co', repo_type='model', repo_id='trisha2710/llama-3-8b-chat-doctor'), pr_revision=None, pr_num=None)

In [31]:
trainer.model.save_pretrained(new_model)
#trainer.model.push_to_hub(new_model, use_temp_dir=False)